In [1]:
"""
Multi-agent research pipeline
  Topic -> Search Agent (Tavily) -> state['search_result']
        -> Reader Agent (BeautifulSoup) -> state['scraped_content']
        -> Writer chain -> Critic chain -> Final output

Setup:
  pip install -r requirements.txt
  export OPENAI_API_KEY=...   TAVILY_API_KEY=...
  python research_pipeline.py "your research topic"
"""
import os
import sys
from typing import TypedDict

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tavily import TavilyClient

from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# AgentExecutor moved to langchain_classic in LangChain 1.x
try:
    from langchain.agents import AgentExecutor, create_tool_calling_agent
except ImportError:
    from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


# ---------------------------------------------------------------- state
class ResearchState(TypedDict, total=False):
    topic: str
    search_result: str
    scraped_content: str
    report: str
    critique: str


def show(label: str, text: str, n: int = 600):
    """Print intermediate state so you can see what each stage produced."""
    print(f"\n{'=' * 20} {label} {'=' * 20}\n{text[:n]}{'...' if len(text) > n else ''}\n")


# ---------------------------------------------------------------- tools
@tool
def web_search_tool(query: str) -> str:
    """Search the web for up-to-date information. Returns titles, URLs and snippets."""
    results = tavily.search(query=query, max_results=5)
    return "\n\n".join(
        f"Title: {r['title']}\nURL: {r['url']}\nSnippet: {r['content']}"
        for r in results.get("results", [])
    )


@tool
def scrape_url_tool(url: str) -> str:
    """Fetch a web page and return its main readable text."""
    try:
        resp = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
    except requests.RequestException as e:
        return f"Could not fetch {url}: {e}"

    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
        tag.decompose()
    text = " ".join(soup.get_text(separator=" ").split())
    return text[:4000]  # keep the context window under control


# ---------------------------------------------------------------- agents
def build_agent(system_msg: str, tools: list) -> AgentExecutor:
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_msg),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ])
    agent = create_tool_calling_agent(llm, tools, prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=6)


search_agent = build_agent(
    "You are a research search agent. Use web_search_tool to find recent, credible "
    "sources on the topic. Return a list of the most relevant results, each with "
    "title, URL and a one-line summary.",
    [web_search_tool],
)

reader_agent = build_agent(
    "You are a reader agent. From the search results given, pick the 2-3 most "
    "relevant URLs and read them with scrape_url_tool. Return the key facts, figures "
    "and arguments you found, grouped by source URL.",
    [scrape_url_tool],
)


# ---------------------------------------------------------------- chains
writer_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a research writer. Write a clear, structured report "
                   "(intro, key findings, conclusion, sources) using ONLY the material provided."),
        ("human", "Topic: {topic}\n\nSearch results:\n{search_result}\n\n"
                  "Scraped content:\n{scraped_content}"),
    ])
    | llm
    | StrOutputParser()
)

critic_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a strict research critic. Check the report against the source "
                   "material: flag unsupported claims, gaps, and bias, then give a score out of 10."),
        ("human", "Topic: {topic}\n\nSource material:\n{scraped_content}\n\nReport:\n{report}"),
    ])
    | llm
    | StrOutputParser()
)


# ---------------------------------------------------------------- pipeline
def run_pipeline(topic: str) -> ResearchState:
    state: ResearchState = {"topic": topic}

    # 1. Search Agent
    out = search_agent.invoke({"input": f"Find recent information about: {topic}"})
    state["search_result"] = out["output"]
    show("state['search_result']", state["search_result"])

    # 2. Reader Agent
    out = reader_agent.invoke({
        "input": f"Topic: {topic}\n\nSearch results:\n{state['search_result']}"
    })
    state["scraped_content"] = out["output"]
    show("state['scraped_content']", state["scraped_content"])

    # 3. Writer chain
    state["report"] = writer_chain.invoke(state)
    show("state['report']", state["report"])

    # 4. Critic chain
    state["critique"] = critic_chain.invoke(state)
    show("state['critique']", state["critique"])

    return state


if __name__ == "__main__":
    topic = " ".join(sys.argv[1:]) or "Latest advances in small language models"
    final = run_pipeline(topic)

    print("\n" + "#" * 60 + "\nFINAL OUTPUT\n" + "#" * 60)
    print(final["report"])
    print("\n--- Critic feedback ---\n")
    print(final["critique"])

/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain_langraph/langchain_langgraph/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




> Entering new AgentExecutor chain...

Invoking: `web_search_tool` with `{'query': 'kernel-v3f4da35f2a8735c8824ba9b4bc4e308fa022ad2e6.json'}`


Title: Medium
URL: https://medium.com/@emanuele.santini.88/introducing-json-integration-in-the-linux-kernel-a-new-feature-explained-f801dcaab034
Snippet: The KJSON library aims to integrate JSON language support directly into the Linux kernel. This integration will enable efficient parsing and handling of JSON data within the kernel itself. The library contains a JSON text parser and a data structure manager to represent the JSON key-value types.

You can get the free GNU library from the JsonOnKernel project: 

Clone the repository on your host and execute the Makefile to build the module.

### How to parse a JSON with the library [...] The use of a JSON language generator integrated in the Linux kernel can offer several advantages, particularly in terms of interoperability and flexibility, making data structure more user-friendly. Here are 